<a href="https://www.kaggle.com/code/halaelbayoumi/hidden-cost-of-ml-projects-resnet-18-on-cifar-10?scriptVersionId=305062643" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ENVIRONMENT SETUP
# ============================================================
!pip install codecarbon -q

import os
import numpy as np
import pandas as pd
import time
import psutil

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import accuracy_score, precision_score, recall_score
from codecarbon import EmissionsTracker

# --- Check GPU ---
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available    : {gpus}")
print(f"TensorFlow version: {tf.__version__}")


# DATA PIPELINE
# ============================================================

# --- Load CIFAR-10 ---
(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()

# --- Normalize pixel values first (keeps data as float32) ---
x_train_full = x_train_full.astype('float32') / 255.0
x_test       = x_test.astype('float32') / 255.0

# --- Normalize using ImageNet mean and std ---
imagenet_mean = np.array([0.485, 0.456, 0.406], dtype='float32')
imagenet_std  = np.array([0.229, 0.224, 0.225], dtype='float32')

x_train_full = (x_train_full - imagenet_mean) / imagenet_std
x_test       = (x_test       - imagenet_mean) / imagenet_std

# --- Split train into train and validation (80/20) ---
train_size = int(0.8 * len(x_train_full))

x_train, x_val = x_train_full[:train_size], x_train_full[train_size:]
y_train, y_val = y_train_full[:train_size], y_train_full[train_size:]

# --- One-hot encode labels ---
y_train_cat = to_categorical(y_train, 10)
y_val_cat   = to_categorical(y_val,   10)
y_test_cat  = to_categorical(y_test,  10)

# --- Build tf.data pipeline with batch-by-batch resizing ---
def preprocess(image, label):
    image = tf.image.resize(image, [224, 224])
    return image, label

BATCH_SIZE = 32

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train_cat))
train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val_cat))
val_ds = val_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test_cat))
test_ds = test_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Training samples  : {len(x_train)}")
print(f"Validation samples: {len(x_val)}")
print(f"Testing samples   : {len(x_test)}")

# BUILD ResNet-18 IN TENSORFLOW
# ============================================================
def residual_block(x, filters, stride=1):
    shortcut = x

    x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


def build_resnet18(num_classes=10):
    inputs = layers.Input(shape=(224, 224, 3))

    x = layers.Conv2D(64, 7, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)

    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128)

    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256)

    x = residual_block(x, 512, stride=2)
    x = residual_block(x, 512)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return Model(inputs, outputs)


# --- Verify model ---
test_model = build_resnet18(num_classes=10)
print(f"\nTotal parameters: {test_model.count_params():,}")


# TRAINING FUNCTION WITH EARLY STOPPING
# ============================================================
def train_model(model, train_ds, val_ds, epochs=30, patience=5):
    early_stop = EarlyStopping(
        monitor              = 'val_loss',
        patience             = patience,
        restore_best_weights = True,
        verbose              = 1
    )

    history = model.fit(
        train_ds,
        validation_data = val_ds,
        epochs          = epochs,
        callbacks       = [early_stop],
        verbose         = 1
    )

    stopped_epoch = len(history.history['loss'])
    return model, stopped_epoch


# EVALUATION FUNCTION
# ============================================================
def evaluate_model(model, test_ds, y_test):
    y_pred_probs = model.predict(test_ds, verbose=0)
    y_pred       = np.argmax(y_pred_probs, axis=1)
    y_true       = y_test.flatten()

    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)

    return accuracy, precision, recall


# 10 RUNS LOOP
# ============================================================
NUM_RUNS = 10
VARIANT  = "ResNet-18 on CIFAR-10 Baseline"
results  = []

for run in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  Starting Run {run} of {NUM_RUNS}")
    print(f"{'='*60}")

    # --- Fresh Model Every Run ---
    model = build_resnet18(num_classes=10)
    model.compile(
        optimizer = Adam(learning_rate=0.001),
        loss      = 'categorical_crossentropy',
        metrics   = ['accuracy']
    )

    # ----------------------------------------------------------
    # TRAINING — Time + Memory + CodeCarbon
    # ----------------------------------------------------------
    process = psutil.Process(os.getpid())

    # Reset GPU memory stats before training
    try:
        tf.config.experimental.reset_memory_stats('GPU:0')
    except:
        pass

    train_tracker = EmissionsTracker(
        project_name       = "resnet18_baseline_train",
        output_file        = "/kaggle/working/codecarbon_train.csv",
        measure_power_secs = 10,
        tracking_mode      = "machine",
        log_level          = "warning"
    )
    train_tracker.start()
    train_start = time.time()

    model, stopped_epoch = train_model(
        model,
        train_ds, val_ds,
        epochs=30, patience=5
    )

    train_end = time.time()
    train_tracker.stop()

    train_time_sec    = round(train_end - train_start, 4)
    cpu_peak_train_mb = round(process.memory_info().rss / (1024 ** 2), 4)

    try:
        gpu_peak_train_mb = round(
            tf.config.experimental.get_memory_info('GPU:0')['peak'] / (1024 ** 2), 4
        )
    except:
        gpu_peak_train_mb = 0.0

    # ----------------------------------------------------------
    # EVALUATION — Time + Memory + CodeCarbon
    # ----------------------------------------------------------
    try:
        tf.config.experimental.reset_memory_stats('GPU:0')
    except:
        pass

    eval_tracker = EmissionsTracker(
        project_name       = "resnet18_baseline_eval",
        output_file        = "/kaggle/working/codecarbon_eval.csv",
        measure_power_secs = 10,
        tracking_mode      = "machine",
        log_level          = "warning"
    )
    eval_tracker.start()
    eval_start = time.time()

    accuracy, precision, recall = evaluate_model(model, test_ds, y_test)

    eval_end = time.time()
    eval_tracker.stop()

    eval_time_sec    = round(eval_end - eval_start, 4)
    cpu_peak_eval_mb = round(process.memory_info().rss / (1024 ** 2), 4)

    try:
        gpu_peak_eval_mb = round(
            tf.config.experimental.get_memory_info('GPU:0')['peak'] / (1024 ** 2), 4
        )
    except:
        gpu_peak_eval_mb = 0.0

    # ----------------------------------------------------------
    # STORE ALL METRICS FOR THIS RUN
    # ----------------------------------------------------------
    run_result = {
        "variant"           : VARIANT,
        "run"               : run,
        "stopped_epoch"     : stopped_epoch,
        "train_time_sec"    : train_time_sec,
        "eval_time_sec"     : eval_time_sec,
        "cpu_peak_train_mb" : cpu_peak_train_mb,
        "gpu_peak_train_mb" : gpu_peak_train_mb,
        "cpu_peak_eval_mb"  : cpu_peak_eval_mb,
        "gpu_peak_eval_mb"  : gpu_peak_eval_mb,
        "test_accuracy"     : round(accuracy,  4),
        "test_precision"    : round(precision, 4),
        "test_recall"       : round(recall,    4),
    }

    results.append(run_result)

    # --- Progressive save after every run ---
    pd.DataFrame(results).to_csv("/kaggle/working/resnet18_cifar10_baseline_results.csv", index=False)

    print(f"\n  Run {run} Complete")
    print(f"  Stopped Epoch  : {stopped_epoch}")
    print(f"  Train Time     : {train_time_sec}s")
    print(f"  Eval Time      : {eval_time_sec}s")
    print(f"  GPU Peak Train : {gpu_peak_train_mb} MB")
    print(f"  GPU Peak Eval  : {gpu_peak_eval_mb} MB")
    print(f"  Accuracy       : {accuracy:.4f}")
    print(f"  Precision      : {precision:.4f}")
    print(f"  Recall         : {recall:.4f}")


# COMPUTE MEAN ROW + EXPORT TO CSV
# ============================================================
df = pd.DataFrame(results)

numeric_cols        = df.drop(columns=["variant", "run"]).mean()
mean_row            = numeric_cols.to_dict()
mean_row["variant"] = VARIANT
mean_row["run"]     = "mean"

df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

cols = [
    "variant",           "run",
    "stopped_epoch",
    "train_time_sec",    "eval_time_sec",
    "cpu_peak_train_mb", "gpu_peak_train_mb",
    "cpu_peak_eval_mb",  "gpu_peak_eval_mb",
    "test_accuracy",     "test_precision",   "test_recall"
]
df = df[cols]

df.to_csv("/kaggle/working/resnet18_cifar10_baseline_results.csv", index=False)

print(f"\n{'='*60}")
print(f"  All {NUM_RUNS} runs completed!")
print(f"  Files saved:")
print(f"  → resnet18_cifar10_baseline_results.csv")
print(f"  → codecarbon_train.csv")
print(f"  → codecarbon_eval.csv")
print(f"{'='*60}")
print(df[["variant", "run", "stopped_epoch", "test_accuracy", "gpu_peak_train_mb"]])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.1/373.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.4 MB/s eta 0:00:00


2026-03-20 13:21:10.576293: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774012870.964369      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774012871.078320      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774012872.033872      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774012872.033933      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774012872.033936      24 computation_placer.cc:177] computation placer alr

GPUs available    : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
TensorFlow version: 2.19.0
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step


I0000 00:00:1774012928.979272      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774012928.982343      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Training samples  : 40000
Validation samples: 10000
Testing samples   : 10000

Total parameters: 11,191,242

  Starting Run 1 of 10


...


  Run 9 Complete
  Stopped Epoch  : 14
  Train Time     : 2215.5026s
  Eval Time      : 12.7142s
  GPU Peak Train : 2931.353 MB
  GPU Peak Eval  : 2234.3506 MB
  Accuracy       : 0.8082
  Precision      : 0.8220
  Recall         : 0.8082

  Starting Run 10 of 10


[codecarbon WARNING @ 17:59:11] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 17:59:11] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 17:59:11] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon WARNING @ 17:59:11] No CPU tracking mode found. Falling back on CPU constant mode.


Epoch 1/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 178s 130ms/step - accuracy: 0.3517 - loss: 1.8353 - val_accuracy: 0.4265 - val_loss: 2.1089
Epoch 2/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 158s 127ms/step - accuracy: 0.6292 - loss: 1.0449 - val_accuracy: 0.6863 - val_loss: 0.9021
Epoch 3/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 157s 125ms/step - accuracy: 0.7281 - loss: 0.7725 - val_accuracy: 0.6796 - val_loss: 0.9748
Epoch 4/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 157s 125ms/step - accuracy: 0.7870 - loss: 0.6150 - val_accuracy: 0.7381 - val_loss: 0.8113
Epoch 5/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 157s 126ms/step - accuracy: 0.8286 - loss: 0.4937 - val_accuracy: 0.7689 - val_loss: 0.7299
Epoch 6/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 157s 125ms/step - accuracy: 0.8656 - loss: 0.3839 - val_accuracy: 0.7721 - val_loss: 0.7607
Epoch 7/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 157s 125ms/step - accuracy: 0.8968 - loss: 0.2900 - val_accuracy: 0.7705 - val_loss: 0.8498
Epoch 8/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 157s 125ms/step - ac

[codecarbon WARNING @ 18:33:35] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 18:33:35] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 18:33:35] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon WARNING @ 18:33:35] No CPU tracking mode found. Falling back on CPU constant mode.



  Run 10 Complete
  Stopped Epoch  : 13
  Train Time     : 2060.6369s
  Eval Time      : 12.0049s
  GPU Peak Train : 3082.6182 MB
  GPU Peak Eval  : 2375.1458 MB
  Accuracy       : 0.8141
  Precision      : 0.8244
  Recall         : 0.8141

  All 10 runs completed!
  Files saved:
  → resnet18_cifar10_baseline_results.csv
  → codecarbon_train.csv
  → codecarbon_eval.csv
                           variant   run  stopped_epoch  test_accuracy  \
0   ResNet-18 on CIFAR-10 Baseline     1           13.0        0.81490   
1   ResNet-18 on CIFAR-10 Baseline     2           11.0        0.77470   
2   ResNet-18 on CIFAR-10 Baseline     3           11.0        0.79550   
3   ResNet-18 on CIFAR-10 Baseline     4           10.0        0.77090   
4   ResNet-18 on CIFAR-10 Baseline     5           13.0        0.80580   
5   ResNet-18 on CIFAR-10 Baseline     6           11.0        0.80470   
6   ResNet-18 on CIFAR-10 Baseline     7           13.0        0.82340   
7   ResNet-18 on CIFAR-10 Baseline 